# Indox — PDF / Documents convert

Simple OpenAPI-style flow: **convert → poll → save**.

| | |
|---|---|
| **What** | `INPUT` → `TARGET` |
| **How** | one POST convert, poll status, GET download |
| **Where** | `output/` next to this notebook |


In [ ]:
# Config
%pip install -q requests

import os
from getpass import getpass
from pathlib import Path

def load_dotenv(path: Path) -> dict:
    out = {}
    if not path.is_file():
        return out
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        out[k.strip()] = v.strip().strip('"').strip("'")
    return out

for p in (Path.cwd().parents[1] / ".env", Path.cwd().parent / ".env", Path("/content/indox-sdk/.env")):
    env = load_dotenv(p)
    if env:
        break
else:
    env = {}

BASE_URL = (
    os.getenv("INDOX_BASE_URL")
    or env.get("INDOX_PUBLIC_BASE_URL")
    or env.get("INDOX_BASE_URL")
    or "https://indox.org"
).rstrip("/")
API_KEY = (os.getenv("INDOX_API_KEY") or env.get("INDOX_API_KEY") or "").strip()
if not API_KEY:
    API_KEY = getpass("INDOX_API_KEY: ").strip()

INPUT = Path('samples/hello.txt')   # what
TARGET = 'pdf'       # convert to
OUT_DIR = Path("output")         # where
OUT_DIR.mkdir(exist_ok=True)

assert INPUT.is_file(), f"missing {INPUT.resolve()}"
print("BASE_URL", BASE_URL)
print("INPUT   ", INPUT.resolve())
print("TARGET  ", TARGET)
print("OUT_DIR ", OUT_DIR.resolve())


In [ ]:
# Convert → poll → save (OpenAPI-style)
import time
import requests

headers = {"Authorization": f"Bearer {API_KEY}"}

# 1) convert
with INPUT.open("rb") as f:
    job = requests.post(
        f"{BASE_URL}/api/v1/pdf_handler/convert/",
        headers=headers,
        data={"target_formats": TARGET},
        files={"file": (INPUT.name, f)},
        timeout=60,
    )
job.raise_for_status()
cid = job.json().get("id") or job.json().get("conversion_id")
print("job", cid)

# 2) poll
status_url = f"{BASE_URL}/api/v1/pdf_handler/conversion/{cid}/"
while True:
    s = requests.get(status_url, headers=headers, timeout=60)
    s.raise_for_status()
    body = s.json()
    state = str(body.get("status") or "").lower()
    print("status", state)
    if state in {"completed", "success", "failed", "error"}:
        break
    time.sleep(1)

if state not in {"completed", "success"}:
    raise SystemExit(body)

# 4) download + save (URL from status payload)
dl_path = body.get("download_url") or (body.get("output") or {}).get(TARGET)
if not dl_path:
    raise SystemExit(f"no download_url in status: {body}")
dl_url = dl_path if str(dl_path).startswith("http") else f"{BASE_URL}{dl_path}"
dl = requests.get(dl_url, headers=headers, timeout=120)
dl.raise_for_status()
out = OUT_DIR / f"{INPUT.stem}.{TARGET}"
out.write_bytes(dl.content)
print("saved", out.resolve(), f"({out.stat().st_size} bytes)")

